# Step 10 – Study area map (paper Figure 1)
CPU runtime is fine, about 2 minutes. Saves `fig0_study_area.png` next to the other figures in Drive.

In [ ]:
PROJECT = 'your-ee-project-id'   # <-- your Earth Engine Cloud project ID
import numpy as np, matplotlib.pyplot as plt, ee, os
from google.colab import drive
drive.mount('/content/drive')
FIG = '/content/drive/MyDrive/Paper2_flood/09_exposure_figures/figures'; os.makedirs(FIG, exist_ok=True)
ee.Authenticate(); ee.Initialize(project=PROJECT)
INK, INK2 = '#0b0b0b', '#52514e'
plt.rcParams.update({'font.size': 9, 'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})

In [ ]:
# Figure 0 - study area map (hillshade, country borders, major rivers, both study boxes)
import geopandas as gpd
from matplotlib.patches import Rectangle
BB = [79.9, 25.9, 88.4, 30.6]                       # map extent: lon/lat
RES = 0.01                                           # ~1 km pixels
nx, ny = int((BB[2] - BB[0]) / RES), int((BB[3] - BB[1]) / RES)
dem = ee.Image('CGIAR/SRTM90_V4').select('elevation')
hs = ee.Terrain.hillshade(dem.multiply(3)).rename('hs').unmask(255).toUint8()   # x3 exaggeration for 1 km pixels
grid = {'dimensions': {'width': nx, 'height': ny}, 'crsCode': 'EPSG:4326',
        'affineTransform': {'scaleX': RES, 'shearX': 0, 'translateX': BB[0], 'shearY': 0, 'scaleY': -RES, 'translateY': BB[3]}}
shade = ee.data.computePixels({'expression': hs, 'fileFormat': 'NUMPY_NDARRAY', 'grid': grid})['hs']
elev = ee.data.computePixels({'expression': dem.unmask(0).toInt16(), 'fileFormat': 'NUMPY_NDARRAY', 'grid': grid})['elevation']

region = ee.Geometry.Rectangle(BB)
def fc_to_gdf(fc):
    return gpd.GeoDataFrame.from_features(fc.getInfo()['features'], crs='EPSG:4326')
countries = fc_to_gdf(ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017').filterBounds(region).select(['country_na']))
try:
    rivers = fc_to_gdf(ee.FeatureCollection('WWF/HydroSHEDS/v1/FreeFlowingRivers').filterBounds(region)
                       .filter(ee.Filter.gte('UPLAND_SKM', 20000)).select(['UPLAND_SKM']))
except Exception as e:
    print('rivers skipped:', str(e)[:120]); rivers = None

fig, ax = plt.subplots(figsize=(9, 5.2))
ext = [BB[0], BB[2], BB[1], BB[3]]
ax.imshow(np.where(elev > 0, elev, np.nan), extent=ext, cmap='Greys', vmin=-3000, vmax=9000, zorder=0)
ax.imshow(shade, extent=ext, cmap='gray', alpha=0.45, zorder=1)
nepal = countries[countries.country_na == 'Nepal']
countries.boundary.plot(ax=ax, color=INK2, linewidth=0.6, zorder=3)
nepal.boundary.plot(ax=ax, color=INK, linewidth=1.4, zorder=4)
if rivers is not None and len(rivers):
    rivers.plot(ax=ax, color='#2a78d6', linewidth=(rivers.UPLAND_SKM / 60000).clip(0.5, 2.0), zorder=5)
for (x0, y0, x1, y1), lab, dx, dy in [((85.20, 27.58, 85.48, 27.78), 'Kathmandu Valley\n(steep, urban)', -1.9, 0.55),
                                      ((86.85, 26.45, 87.15, 26.75), 'Koshi Terai\n(flat floodplain)', 0.35, -0.05)]:
    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='#eb6834', linewidth=2, zorder=6))
    ax.annotate(lab, xy=(x0 if dx < 0 else x1, (y0 + y1) / 2), xytext=(x0 + dx, y1 + dy), fontsize=8.5, color=INK,
                arrowprops={'arrowstyle': '-', 'color': INK2, 'linewidth': 0.8}, zorder=7,
                bbox={'boxstyle': 'round,pad=0.25', 'fc': 'white', 'ec': 'none', 'alpha': 0.85})
for x, y, t in [(82.6, 28.9, 'NEPAL'), (81.0, 26.6, 'INDIA'), (84.5, 30.35, 'CHINA')]:
    ax.text(x, y, t, fontsize=9, color=INK2, ha='center', zorder=7, fontweight='bold')
ax.plot(87.0, 26.52, marker='v', color=INK, markersize=5, zorder=8)
ax.text(86.92, 26.25, 'Koshi Barrage', fontsize=7.5, color=INK, ha='right', zorder=8,
        bbox={'boxstyle': 'round,pad=0.2', 'fc': 'white', 'ec': 'none', 'alpha': 0.85})
ax.set_xlim(BB[0], BB[2]); ax.set_ylim(BB[1], BB[3]); ax.set_aspect(1 / np.cos(np.radians(28.4)))
ax.grid(False); ax.spines[['top', 'right']].set_visible(True)
ax.set_xticks(range(80, 89, 2)); ax.set_xticklabels([f'{v}°E' for v in range(80, 89, 2)])
ax.set_yticks(range(26, 31)); ax.set_yticklabels([f'{v}°N' for v in range(26, 31)])
ax.set_title('Study areas: September 2024 Nepal floods', loc='left', fontsize=10, color=INK)
ax.set_xlabel(''); ax.set_ylabel('')
ax.text(1.0, -0.09, 'Hillshade: SRTM 90 m · Rivers: HydroSHEDS (upstream area ≥ 20,000 km²) · Borders: LSIB 2017',
        transform=ax.transAxes, ha='right', va='top', fontsize=6.5, color=INK2)
plt.savefig(f'{FIG}/fig0_study_area.png'); plt.show()
